# Notebook 03: Golden Evaluation Set Construction & Data Leakage Audit

This notebook verifies the 200-item static Golden Evaluation Set, 50-item Human Calibration set, Data Leakage Guard zero-overlap check, and benchmark results.


In [ ]:
import json, os

golden_path = "golden/golden_set.jsonl"
if os.path.exists(golden_path):
    with open(golden_path, "r", encoding="utf-8") as f:
        items = [json.loads(line) for line in f]
    print(f"Loaded {len(items)} Golden Evaluation items.")
    esc_count = sum(1 for i in items if i["gold_escalate"])
    print(f"Escalated: {esc_count} ({esc_count/len(items)*100:.1f}%), Auto-Handle: {len(items)-esc_count}")
else:
    print("Golden set file not found.")


## 2. Data Leakage Guard Verification
Verifying zero overlap between 200 Golden Set items and 19,800 training/indexing threads.


In [ ]:
processed_path = "data/processed/amazonhelp_threads.jsonl"
if os.path.exists(golden_path) and os.path.exists(processed_path):
    with open(golden_path, "r", encoding="utf-8") as f:
        g_ids = {json.loads(line)["conversation_id"] for line in f}
    with open(processed_path, "r", encoding="utf-8") as f:
        p_threads = [json.loads(line) for line in f]
    train_ids = {t["conversation_id"] for t in p_threads if t["conversation_id"] not in g_ids}
    overlap = g_ids.intersection(train_ids)
    print(f"Golden Set IDs: {len(g_ids)}")
    print(f"Training Set IDs: {len(train_ids)}")
    print(f"ID Overlap Count: {len(overlap)}")
    assert len(overlap) == 0, "Data Leakage Violation!"
    print("Data Leakage Guard PASSED — 0% overlap between golden set and training set.")


## 3. Benchmark Evaluation Summary
Displaying ground truth evaluation metrics from `results/evaluation_results.json`.


In [ ]:
res_path = "results/evaluation_results.json"
if os.path.exists(res_path):
    with open(res_path, "r", encoding="utf-8") as f:
        res = json.load(f)
    print("Resolved Brand:", res.get("resolved_brand_handle"))
    print("Benchmark Results:")
    print(json.dumps(res.get("benchmark_results"), indent=2))
else:
    print("Results JSON not present.")
